In [20]:
from together import Together
import os
import random
import json
import re

API_KEY = "4f0440e2cc8baa37f8017270d398124bd4e477459f3500c74290591509b2e6b3"

# ============================================================================
# Helper functions
# ============================================================================

# Messages: yes contains messages
def call_llm(client, model, temperature, messages, logprobs=True, top_logprobs=5):
    """Call LLM and get response"""
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        logprobs=logprobs,
        top_logprobs=top_logprobs
    )
    return response.choices[0].message.content

def print_simulation_header(game, num_turns, num_agents, memory_capacity, agent_biases):
    """Print simulation configuration header"""
    print("=" * 80)
    print(f"SIMULATION: {game.__class__.__name__}")
    print("=" * 80)
    print(f"Number of rounds: {num_turns}")
    print(f"Number of agents: {num_agents}")
    print(f"Memory capacity: {memory_capacity}")
    if agent_biases:
        print(f"Agent biases: {agent_biases}")
    print("-" * 80)

# ============================================================================
# CORE INFRASTRUCTURE
# ============================================================================

class Agent:
    """Represents a single LLM agent with memory capacity"""

    def __init__(self, agent_id, model, temperature, client, memory_capacity, initial_bias, system_prompt=None): #system prompt is none, so it can be set later. 
        self.agent_id = agent_id
        self.model = model
        self.temperature = temperature
        self.client = client
        self.memory_capacity = memory_capacity
        self.messages = []
        self.initial_bias = initial_bias
        self.system_prompt = system_prompt
    

    # Response with messages
    def respond(self, prompt):
        """Respond to a prompt with the LLM. The truncation effectuates
        a short term memory effect; earlier interactions are forgotten. Thus introduces recency bias; 
        recent interactions have more impact than older ones. Remove if unwanted"""
        # Truncate oldest messages if memory is full (but keep system prompt)
        if len(self.messages) > self.memory_capacity * 2 + 1:  # *2 for user and assistant messages, +1 for system prompt
            # Keep system prompt (first message) and last N messages
            self.messages = [self.messages[0]] + self.messages[-(self.memory_capacity * 2):]
        
        self.messages.append({"role": "user", "content": prompt})

        # Call the LLM
        response = call_llm(self.client, self.model, self.temperature, self.messages)
        self.messages.append({"role": "assistant", "content": response})
        return response

class SimulationData:
    """Centralized state management for multi-agent conversations"""

    def __init__(self):
        self.agents = {}
        self.conversation_history = []
        self.game_data = {}

    def add_agent(self, agent_id, agent):
        self.agents[agent_id] = agent

    def get_agent_messages(self, agent_id):
        return self.agents[agent_id].messages

    def save_state(self, filepath):
        state = {
            "conversation_history": self.conversation_history,
            "game_data": self.game_data,
            "agents": {
                agent_id: {
                    "agent_id": agent.agent_id,
                    "model": agent.model,
                    "memory_capacity": agent.memory_capacity,
                    "initial_bias": agent.initial_bias,
                    "system_prompt": agent.system_prompt,
                    "messages": agent.messages
                }
                for agent_id, agent in self.agents.items()
            }
        }
        with open(filepath, 'w') as f:
            json.dump(state, f, indent=2)

# ============================================================================
# GAME INTERFACE (Base Class)
# ============================================================================

class Game:
    """Base class for all games"""
    
    def __init__(self):
        pass
    
    def get_system_prompt(self, agent_id, agent):
        """Return the system prompt for an agent (game rules/context)"""
        raise NotImplementedError
    
    def get_round_1_prompt(self, agent_id, agent):
        """Return the initial user prompt for first turn"""
        raise NotImplementedError
    
    def get_later_round_prompt(self, agent_id, turn, sim_data, last_responses):
        """Return the prompt for an agent on subsequent turns"""
        raise NotImplementedError
    
    def process_turn(self, turn, agent_responses, sim_data):
        """Process the responses from all agents and update game state"""
        raise NotImplementedError
    
    def print_turn_summary(self, turn, agent_responses, sim_data):
        """Print summary of what happened this turn"""
        pass
    
    def print_game_summary(self, sim_data):
        """Print final game summary"""
        pass

# ============================================================================
# TRUST GAME IMPLEMENTATION
# ============================================================================

class TrustGame(Game):
    """Trust/Investment game with sequential moves"""
    
    def __init__(self, endowment, multiplier):
        super().__init__()
        self.endowment = endowment
        self.multiplier = multiplier
        self.investor_id = "Agent_1"
        self.trustee_id = "Agent_2"
    
    def get_move_order(self, turn, sim_data):
        """Define who moves in what order this turn"""
        return [self.investor_id, self.trustee_id]
    
    def get_system_prompt(self, agent_id, agent):
        """Role-specific rules"""
        if agent_id == self.investor_id:
            return f"""You are the INVESTOR in a trust game.

Each round you receive ${self.endowment}.
1. You decide how much to SEND (0-${self.endowment})
2. Amount is multiplied by {self.multiplier}x
3. Trustee decides how much to RETURN
4. You keep: ${self.endowment} - sent + returned

IMPORTANT: Always respond in format:
{{'send': <amount>, 'reason': <reasoning>}}"""
        else:
            return f"""You are the TRUSTEE in a trust game.

Each round:
1. Investor receives ${self.endowment} and decides how much to SEND (0-${self.endowment})
2. Amount sent is multiplied by {self.multiplier}x and you receive it
3. You decide how much to RETURN
4. You keep: received - returned
5. Investor keeps: ${self.endowment} - sent + returned

IMPORTANT: Always respond in format:
{{'return': <amount>, 'reason': <reasoning>}}"""
    
    def get_game_prompt_round_1(self, agent_id, agent):
        """First turn"""
        if agent_id == self.investor_id:
            return f"Round 1: You have ${self.endowment}. How much do you send? (0-{self.endowment})"
        else:
            # Trustee gets prompted after investor, will have pending_sent available
            sent = self.sim_data_ref.game_data.get("pending_sent", 0)
            received = sent * self.multiplier
            percentage = (sent / self.endowment * 100) if self.endowment > 0 else 0
            return f"Round 1: Investor sent ${sent}, that is {percentage:.1f}% of its total endowment. You received ${received}. How much do you return? (0-{received})"
    
    def get_game_prompt_later_round(self, agent_id, turn, sim_data, last_responses):
        """Subsequent turns"""
        if agent_id == self.investor_id:
            last_round = sim_data.conversation_history[-1]
            investor_balance = sim_data.game_data["balances"][self.investor_id]
            
            return f"""Round {turn}

Last round: You sent ${last_round['sent']}, received back ${last_round['returned']}
Your total earnings: ${investor_balance}

You have ${self.endowment}. How much do you send? (0-{self.endowment})"""
        else:
            sent = sim_data.game_data["pending_sent"]
            received = sent * self.multiplier
            trustee_balance = sim_data.game_data["balances"][self.trustee_id]
            percentage = (sent / self.endowment * 100) if self.endowment > 0 else 0
            return f"""Round {turn}

Investor sent ${sent}, that is {percentage:.1f}% of its total endowment. You received ${received}.
Your total earnings: ${trustee_balance}

How much do you return? (0-{received})"""
    
    def process_intermediate_response(self, agent_id, response, turn, sim_data):
        """Called after investor responds, before trustee"""
        if agent_id == self.investor_id:
            sent_amount = self._extract_amount(response, "send")
            sent_amount = max(0, min(sent_amount, self.endowment)) # Clamps to valid range: Forces 0 ≤ amount ≤ endowment (do we want this???))
            sim_data.game_data["pending_sent"] = sent_amount
            # Store reference for get_round_1_prompt to use
            self.sim_data_ref = sim_data
    
    def process_turn(self, turn, agent_responses, sim_data):
        """Process complete turn (both moves done)"""
        # Initialize on first turn
        if "balances" not in sim_data.game_data:
            sim_data.game_data["balances"] = {self.investor_id: 0, self.trustee_id: 0}
            sim_data.game_data["pending_sent"] = 0
            self.sim_data_ref = sim_data
        
        # Extract amounts
        sent = self._extract_amount(agent_responses[self.investor_id], "send")
        sent = max(0, min(sent, self.endowment)) # Clamps to valid range: Forces 0 ≤ amount ≤ endowment (do we want this???))
        
        returned = self._extract_amount(agent_responses[self.trustee_id], "return")
        received = sent * self.multiplier
        returned = max(0, min(returned, received)) # Clamps to valid range: Forces 0 ≤ amount ≤ received (do we want this???))
        
        # Calculate payoffs
        investor_payoff = (self.endowment - sent) + returned
        trustee_payoff = received - returned
        
        # Update balances
        sim_data.game_data["balances"][self.investor_id] += investor_payoff
        sim_data.game_data["balances"][self.trustee_id] += trustee_payoff
        
        # Record round
        sim_data.conversation_history.append({
            "round": turn,
            "sent": sent,
            "received": received,
            "returned": returned,
            "investor_payoff": investor_payoff,
            "trustee_payoff": trustee_payoff,
            "balances": dict(sim_data.game_data["balances"]),
            "myths": {}
        })
        
        return {
            self.investor_id: {"sent": sent},
            self.trustee_id: {"returned": returned}
        }
    
    # extraction works by finding '{key}': <number> in the response. 
    def _extract_amount(self, response, key):
        """Extract number from response"""
        pattern = rf"'{key}':\s*(\d+\.?\d*)|" + rf'"{key}":\s*(\d+\.?\d*)'
        match = re.search(pattern, response, re.IGNORECASE)
        if match:
            return float(match.group(1) or match.group(2))
        raise ValueError(f"Could not extract {key} from: {response[:200]}")
    
    def print_turn_summary(self, turn, agent_responses, sim_data):
        """Print round summary"""
        entry = sim_data.conversation_history[-1]
        print(f"\n{'*' * 80}")
        print(f"ROUND {turn} COMPLETE")
        print(f"  Sent: ${entry['sent']} → Received: ${entry['received']} → Returned: ${entry['returned']}")
        print(f"  Payoffs: Investor ${entry['investor_payoff']}, Trustee ${entry['trustee_payoff']}")
        print(f"  Cumulative: Investor ${entry['balances'][self.investor_id]}, Trustee ${entry['balances'][self.trustee_id]}")
        print(f"{'*' * 80}")
    
    def print_game_summary(self, sim_data):
        """Final summary"""
        num_rounds = len(sim_data.conversation_history)
        
        print("\n" + "=" * 80)
        print("GAME SUMMARY")
        print("=" * 80)
        
        total_sent = sum(r["sent"] for r in sim_data.conversation_history)
        total_returned = sum(r["returned"] for r in sim_data.conversation_history)
        avg_sent = total_sent / num_rounds
        avg_returned = total_returned / num_rounds
        
        print(f"\nRounds: {num_rounds}")
        print(f"Avg sent: ${avg_sent:.2f}/{self.endowment}")
        print(f"Avg returned: ${avg_returned:.2f}")
        print(f"Final earnings: Investor ${sim_data.game_data['balances'][self.investor_id]}, Trustee ${sim_data.game_data['balances'][self.trustee_id]}")

# ============================================================================
# MYTH WRITING
# ============================================================================

class MythWriter:
    """Handles myth writing functionality, separate from game logic"""
    
    def __init__(self, myth_topic):
        self.myth_topic = myth_topic
    
    def get_myth_prompt_round_1(self, agent_id, turn, sim_data):
        """Generate prompt for myth writing
            1. Get last entry.
            2. Get agent's choice from last entry.
            3. Get coordination result from last entry.
            4. Construct prompt.
            note: coordination result and agent choice are not used in the prompt atm.
        """
        last_entry = sim_data.conversation_history[-1]
        # agent_choice = last_entry["choices"][agent_id]
        # coordination_result = last_entry["coordination"]

            # OLD MYTH PROMPT
            #         return f"""Write a myth about {self.myth_topic}. 
            # Write 200 words. 
            # Format exactly:
            # Myth: [your story here]."""
        return f"""Write a myth about {self.myth_topic}. Write 200 words."""

    def get_myth_prompt_round_later(self, agent_id, turn, sim_data):
        """Generate prompt for myth writing with the agent's previous myth"""
        # Get this agent's myth from previous round
        last_myth = ""
        other_agent_myth = ""
        # Find previous turn's entry: loops through all entries in conversation history. Checks the round number and if the entry has a "myths" key.
        # If the entry has a "myths" key, it checks if the agent_id is in the entry["myths"] dictionary. If it is, it gets the myth.
        # If the entry has a "myths" key, it also gets the other agent's myth.
        for entry in sim_data.conversation_history:
            if entry["round"] == turn - 1 and "myths" in entry:
                if agent_id in entry["myths"]:
                    last_myth = entry["myths"][agent_id]
                
                # Get the other agent's myth
                for other_agent_id, myth in entry["myths"].items():
                    if other_agent_id != agent_id:
                        other_agent_myth = myth
                        break
                break
        
        if last_myth:
            if other_agent_myth:
                prompt = f"""Here is the myth you wrote in the previous round: 
{last_myth}

Here is the myth the other agent wrote in the previous round:
{other_agent_myth}

Write your own myth about {self.myth_topic}. 
Use your previous myth as inspiration, but adapt it in your own way. 
Write 200 words. 
Format exactly:
Myth: [your story here]."""
            else:
                raise ValueError(f"OTHER AGENT MYTH ERROR:No previous myth found for {agent_id} (other agent) in round {turn - 1}. Cannot generate later round prompt.")
        
        else:
            raise ValueError(f"NO SELF MYTH ERROR: No previous myth found for {agent_id} (you/self agent) in round {turn - 1}. Cannot generate later round prompt.")
        
        return prompt


    def process_myths(self, turn, agent_myths, sim_data):
        """Store the myths written by agents. 
            1. Find the entry for this turn.
            2. Add myths to the entry.
        """
        for entry in sim_data.conversation_history:
            if entry["round"] == turn:
                entry["myths"] = agent_myths
                break
            
# ============================================================================
# SIMULATION LOOP (Modified for sequential moves)
# ============================================================================

def run_simulation(game, model, temperature, num_turns, num_agents, memory_capacity, agent_biases, myth_writer):
    """
    Run a multi-agent simulation with any game.
    Now supports sequential moves within a turn.
    """
    client = Together(api_key=API_KEY)
    sim_data = SimulationData()
    
    # Initialize agents
    for i in range(num_agents):
        agent_id = f"Agent_{i+1}"
        bias = agent_biases[i] if agent_biases and i < len(agent_biases) else None
        agent = Agent(agent_id, model, temperature, client, memory_capacity=memory_capacity, initial_bias=bias)
        system_prompt = game.get_system_prompt(agent_id, agent)
        agent.system_prompt = system_prompt
        agent.messages.append({"role": "system", "content": system_prompt})
        sim_data.add_agent(agent_id, agent)
    
    print_simulation_header(game, num_turns, num_agents, memory_capacity, agent_biases)
    
    last_responses = {}
    
    # Main simulation loop
    for turn in range(1, num_turns + 1):
        print("\n" + "=" * 80)
        print(f"ROUND {turn}")
        print("=" * 80)
        
        agent_responses = {}
        
        # PHASE 1: GAME PLAY
        print("\n--- PHASE 1: GAME PLAY ---")
        
        # Sequential moves; because game has sequential move order. 
        move_order = game.get_move_order(turn, sim_data)
        
        for agent_id in move_order:
            agent = sim_data.agents[agent_id]
            
            if turn == 1:
                prompt = game.get_game_prompt_round_1(agent_id, agent)
            else:
                prompt = game.get_game_prompt_later_round(agent_id, turn, sim_data, last_responses)
            
            response = agent.respond(prompt)
            agent_responses[agent_id] = response
            
            # Format agent_id for print display with role
            print(f"\n{'Agent 1 (Investor)' if agent_id == game.investor_id else 'Agent 2 (Trustee)'} prompt: {prompt}")
            print(f"{'Agent 1 (Investor)' if agent_id == game.investor_id else 'Agent 2 (Trustee)'} response: {response}")

            
            # Allow game to update state after each move (for sequential games)
            if hasattr(game, 'process_intermediate_response'):
                game.process_intermediate_response(agent_id, response, turn, sim_data)
        # else:
        #     # Simultaneous moves (original behavior)
        #     for agent_id, agent in sim_data.agents.items():
        #         if turn == 1:
        #             prompt = game.get_round_1_prompt(agent_id, agent)
        #         else:
        #             prompt = game.get_later_round_prompt(agent_id, turn, sim_data, last_responses)
                
        #         response = agent.respond(prompt)
        #         agent_responses[agent_id] = response
                
        #         print(f"\n{agent_id} prompt: {prompt}")
        #         print(f"{agent_id} response: {response}")
        
        # Process turn with game logic
        last_responses = game.process_turn(turn, agent_responses, sim_data)
        
        # PHASE 2: MYTH WRITING 
        print("\n--- PHASE 2: MYTH WRITING ---")
        agent_myths = {}
        
        # Use same order as game moves for consistency
        move_order = game.get_move_order(turn, sim_data)
        for agent_id in move_order:
            agent = sim_data.agents[agent_id]
            # Get the myth prompt and response
            if turn == 1:
                myth_prompt = myth_writer.get_myth_prompt_round_1(agent_id, turn, sim_data)
            else:
                myth_prompt = myth_writer.get_myth_prompt_round_later(agent_id, turn, sim_data)
            myth_response = agent.respond(myth_prompt)
            agent_myths[agent_id] = myth_response
            
            # Format agent_id for print display with role
            print(f"\n{'Agent 1 (Investor)' if agent_id == game.investor_id else 'Agent 2 (Trustee)'} myth prompt:\n{myth_prompt}")
            print(f"\n{'Agent 1 (Investor)' if agent_id == game.investor_id else 'Agent 2 (Trustee)'} myth response:\n{myth_response}")
            #print(f"\n{'Agent 1 (Investor)' if agent_id == game.investor_id else 'Agent 2 (Trustee)'} myth response:\n{myth_response[:100]}...")

        myth_writer.process_myths(turn, agent_myths, sim_data)
        
        # Print turn summary
        game.print_turn_summary(turn, agent_responses, sim_data)
        
        # Print myths
        if sim_data.conversation_history[-1].get("myths"):
            print(f"\n{'~' * 80}")
            print("MYTHS WRITTEN THIS ROUND:")
            print(f"{'~' * 80}")
            for agent_id, myth in sim_data.conversation_history[-1]["myths"].items():
                print(f"\n{'Agent 1 (Investor)' if agent_id == game.investor_id else 'Agent 2 (Trustee)'}:")
                print(myth)
                print("-" * 40)
    
    game.print_game_summary(sim_data)
    return sim_data

# ============================================================================
# USAGE
# ============================================================================

if __name__ == "__main__":
    # To change games, just swap this line:
    game = TrustGame(endowment=5, multiplier=3)   # initial endowment should be 5, otherwise trust game does not make sense. 
    myth_writer = MythWriter(myth_topic="") 
    model = "meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo" 
    
    # Run simulation with different initial biases
    sim_data = run_simulation(
        game,
        model, 
        temperature=0.8,
        num_turns=50, 
        num_agents=2,
        memory_capacity=3,
        agent_biases="",
        myth_writer=myth_writer
    )
    
    base_path = "data/json/main_loop/"
    # Save state
    sim_data.save_state(base_path + "3_Trust_simulation_state.json")
    print(f"\n✓ Simulation state saved to 3_Trust_simulation_state.json")

SIMULATION: TrustGame
Number of rounds: 50
Number of agents: 2
Memory capacity: 3
--------------------------------------------------------------------------------

ROUND 1

--- PHASE 1: GAME PLAY ---

Agent 1 (Investor) prompt: Round 1: You have $5. How much do you send? (0-5)
Agent 1 (Investor) response: {'send': 2, 'reason': 'I'll start with a moderate amount to test the trustee's trustworthiness while still allowing for potential gains.'}

Agent 2 (Trustee) prompt: Round 1: Investor sent $2.0, that is 40.0% of its total endowment. You received $6.0. How much do you return? (0-6.0)
Agent 2 (Trustee) response: {'return': 4.0, 'reason': 'The investor has shown a high level of trust by sending 40% of their endowment, so I will reciprocate with a generous return, demonstrating my own trustworthiness and encouraging the investor to send more in the next round.'}

--- PHASE 2: MYTH WRITING ---

Agent 1 (Investor) myth prompt:
Write a myth about . Write 200 words.

Agent 1 (Investor) myth r